In [1]:
import numpy as np
from neural_recording_chain.noise import thermal_noise_uV, background_noise_uV, slow_wave_uV
from neural_recording_chain.filter import bandpass_filter
from neural_recording_chain.adc import digitise, to_codes
from neural_recording_chain.detector import detect_spikes, detect_spikes_on_chip
from neural_recording_chain.scoring import score_detection

In [2]:
file_names = ["simulation_60s.npz", "simulation_60s_seed1.npz", "simulation_60s_input3.npz"]
names = ["A (tuning)", "B (test)", "C (test, stronger input)"]
thermal_seeds = [0, 2, 4]
background_seeds = [1, 3, 5]

dt_ms = 0.05
samples_to_settle = 32768
settle_time_ms = samples_to_settle * dt_ms     # 1638.4 ms
min_gap_samples = round(3.0 / dt_ms)           # 60 samples = 3 ms

for k in range(3):
    data = np.load("../data/processed/" + file_names[k])
    time_ms = data["time_ms"]
    spike_times = data["spike_times"]
    spike_heights = data["spike_heights"]

    true_times = []
    tall_times = []
    for j in range(len(spike_times)):
        if spike_times[j] >= settle_time_ms:
            true_times.append(spike_times[j])
            if spike_heights[j] > 0:
                tall_times.append(spike_times[j])

    # the same signal for both detectors: noise, filter, 8 bits at +-500 uV
    n_samples = len(data["v_electrode_uV"])
    recording = (data["v_electrode_uV"]
                 + thermal_noise_uV(n_samples, dt_ms=dt_ms, seed=thermal_seeds[k])
                 + background_noise_uV(n_samples, dt_ms=dt_ms, seed=background_seeds[k])
                 + slow_wave_uV(n_samples, dt_ms=dt_ms))
    filtered = bandpass_filter(recording, dt_ms=dt_ms)

    # floating-point detector, on microvolts
    chip_signal_uV, step_uV = digitise(filtered, n_bits=8, input_range_uV=500.0)
    float_detections, float_threshold_uV = detect_spikes(chip_signal_uV, time_ms,
                                                         threshold_factor=4.5, min_gap_ms=3.0)
    float_times = []
    for detection_time in float_detections:
        if detection_time >= settle_time_ms:
            float_times.append(detection_time)

    # chip detector, on whole-number codes, sample numbers turned back into times
    codes, step_uV = to_codes(filtered, n_bits=8, input_range_uV=500.0)
    chip_samples = detect_spikes_on_chip(codes, min_gap_samples, samples_to_settle=samples_to_settle)
    chip_times = time_ms[chip_samples]

    print("recording", names[k])
    detector_names = ["  floating-point", "  chip          "]
    detector_times = [float_times, chip_times]
    for d in range(2):
        tall_hits, tall_misses, unused = score_detection(tall_times, detector_times[d])
        hits, misses, false_alarms = score_detection(true_times, detector_times[d])
        small_found = hits - tall_hits
        print(detector_names[d], "| tall:", tall_hits, "of", len(tall_times),
              "| small:", small_found, "of", len(true_times) - len(tall_times),
              "| false alarms:", false_alarms)

recording A (tuning)
  floating-point | tall: 201 of 203 | small: 41 of 319 | false alarms: 6
  chip           | tall: 201 of 203 | small: 38 of 319 | false alarms: 7
recording B (test)
  floating-point | tall: 160 of 160 | small: 38 of 283 | false alarms: 4
  chip           | tall: 160 of 160 | small: 38 of 283 | false alarms: 5
recording C (test, stronger input)
  floating-point | tall: 1062 of 1071 | small: 241 of 934 | false alarms: 11
  chip           | tall: 1059 of 1071 | small: 167 of 934 | false alarms: 2
